# 04 — CNN profunda + balanceo de clases

Mismo notebook que 03 (arquitectura de tres bloques, señal filtrada), añadiendo `pos_weight` + focal loss para tratar el desbalanceo entre clases. El umbral de decisión se mantiene fijo en 0.5 — optimizarlo por clase es el último peldaño, en 05. Aísla el efecto del balanceo frente a la arquitectura, que ya se evaluó en el salto 02→03.

## 1. Entorno, semillas y librerías

In [1]:
import sys, os, random, hashlib, json
from pathlib import Path

import numpy as np
import pandas as pd
import ast

import wfdb
from scipy import signal as sp_signal
from scipy.signal import sosfiltfilt, iirnotch, tf2sos

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.regularizers import l2

import matplotlib.pyplot as plt

print(sys.executable)
print("TensorFlow:", tf.__version__)

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


C:\Users\Usuario\anaconda3\envs\tfg\python.exe
TensorFlow: 2.16.1


## 2. Configuración general


In [3]:
NOTEBOOK_NAME = "04_cnn_profunda_balanceo"
PREP_VERSION  = "con_filtrado"     # reutiliza el mismo caché que 02 y 03, la señal no cambia

DATA_DIR  = Path(r"C:\Users\Usuario\Desktop\TFG\ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3")
CACHE_DIR = DATA_DIR / "cache_v2" / PREP_VERSION
CACHE_DIR.mkdir(parents=True, exist_ok=True)

MODEL_DIR = Path("./outputs/models"); MODEL_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR   = Path("./outputs/figures"); FIG_DIR.mkdir(parents=True, exist_ok=True)
COMPARISON_CSV = MODEL_DIR / "model_comparison.csv"

FS = 100
SUPER_CLASSES = ["NORM", "MI", "STTC", "HYP", "CD"]
CLASS_MAP_HUMAN = {
    "NORM": "Normal", "MI": "Infarto", "STTC": "Isquemia/ST-T",
    "HYP": "Hipertrofia", "CD": "Conducción (bloqueos)",
}

BANDPASS_LOW  = 0.5
BANDPASS_HIGH = 40.0
NOTCH_FREQ    = 50.0
NOTCH_Q       = 30.0

STFT_NPERSEG  = 64
STFT_NOVERLAP = 48
SPEC_SHAPE    = (33, 64)

L2_REG = 1e-4   # fuerza de la regularización L2
GAMMA  = 1.0    # factor de enfoque de la focal loss


## 3. Carga de metadatos y etiquetas multi-etiqueta

In [4]:
df  = pd.read_csv(DATA_DIR / "ptbxl_database.csv", index_col="ecg_id")
scp = pd.read_csv(DATA_DIR / "scp_statements.csv", index_col=0)
df.columns  = df.columns.str.strip()
scp.columns = scp.columns.str.strip()

scp_super = (scp[scp["diagnostic_class"].notna()][["diagnostic_class"]]
             .rename(columns={"diagnostic_class": "superclass"}))

def extract_superclasses(scp_codes_str):
    """Convierte el string tipo diccionario de scp_codes en la lista de superclases presentes."""
    if pd.isna(scp_codes_str):
        return []
    try:
        d = ast.literal_eval(scp_codes_str)
        if not isinstance(d, dict):
            return []
    except Exception:
        return []
    return list({scp_super.loc[k, "superclass"] for k in d.keys() if k in scp_super.index})

def multilabel_vector(classes):
    """Convierte la lista de superclases en un vector binario de tamaño 5."""
    s = set(classes)
    return np.array([1 if c in s else 0 for c in SUPER_CLASSES], dtype=np.int32)

df["superclasses"] = df["scp_codes"].apply(extract_superclasses)
df = df[df["superclasses"].apply(len) > 0].reset_index()
df["target_vec"] = df["superclasses"].apply(multilabel_vector)


## 4. Split por paciente

In [5]:
train_df = df[df["strat_fold"].isin(range(1, 9))].reset_index(drop=True)
val_df   = df[df["strat_fold"] == 9].reset_index(drop=True)
test_df  = df[df["strat_fold"] == 10].reset_index(drop=True)

assert set(train_df.patient_id) & set(val_df.patient_id) == set()
assert set(train_df.patient_id) & set(test_df.patient_id) == set()
assert set(val_df.patient_id)  & set(test_df.patient_id)  == set()

print("Tamaños:", len(train_df), len(val_df), len(test_df))


Tamaños: 17084 2146 2158


## 5. Preprocesado de la señal ECG (limpieza)

Idéntico al `preprocess_ecg` del notebook principal: filtro paso-banda de fase cero (0.5-40Hz), notch a 50Hz, y normalización z-score por derivación. Es la única función nueva respecto a 01.

In [6]:
def is_valid_signal(x, flat_std_thresh=1e-6):
    """Descarta derivaciones corruptas: con NaN/Inf o completamente planas (fallo de electrodo)."""
    if np.any(np.isnan(x)) or np.any(np.isinf(x)):
        return False
    return np.std(x) >= flat_std_thresh

_BP_SOS    = sp_signal.butter(4, [BANDPASS_LOW / (FS/2), BANDPASS_HIGH / (FS/2)], btype="band", output="sos")
_NOTCH_SOS = tf2sos(*iirnotch(NOTCH_FREQ / (FS/2), NOTCH_Q))

def clean_lead(x):
    """Limpia una derivación (1D): paso-banda + notch + normalización z-score."""
    x = np.asarray(x, dtype=np.float64)
    if not is_valid_signal(x):
        return None
    x = sosfiltfilt(_BP_SOS, x)
    x = sosfiltfilt(_NOTCH_SOS, x)
    mu, sigma = x.mean(), x.std()
    if sigma < 1e-8:
        return None
    return ((x - mu) / sigma).astype(np.float32)

def preprocess_ecg(raw_signal):
    """Limpia las 12 derivaciones de un ECG. Descarta el registro si más de la mitad fallan."""
    cleaned = np.zeros_like(raw_signal, dtype=np.float32)
    n_bad = 0
    for i in range(raw_signal.shape[1]):
        c = clean_lead(raw_signal[:, i])
        cleaned[:, i] = c if c is not None else 0.0
        n_bad += (c is None)
    return None if n_bad > raw_signal.shape[1] // 2 else cleaned


## 6. Extracción de características: espectrograma STFT


In [9]:
def lead_to_spectrogram(lead_signal, fs=FS, nperseg=STFT_NPERSEG, noverlap=STFT_NOVERLAP, out_shape=SPEC_SHAPE):
    """Convierte una derivación 1D en su espectrograma STFT normalizado, con forma fija (F, T)."""
    _, _, Zxx = sp_signal.stft(lead_signal, fs=fs, nperseg=nperseg, noverlap=noverlap)
    mag = np.log1p(np.abs(Zxx))

    F, T = out_shape
    mag = mag[:F, :T]
    if mag.shape != (F, T):
        pad = np.zeros((F, T), dtype=np.float32)
        pad[:mag.shape[0], :mag.shape[1]] = mag
        mag = pad

    mn, mx = mag.min(), mag.max()
    if mx - mn > 1e-8:
        mag = (mag - mn) / (mx - mn)
    return mag.astype(np.float32)

def ecg_to_tensor(signal_2d):
    """signal_2d: (n_muestras, 12) -> tensor (F, T, 12)."""
    specs = [lead_to_spectrogram(signal_2d[:, i]) for i in range(signal_2d.shape[1])]
    return np.stack(specs, axis=-1)


## 7. Generación del dataset con caché en disco


In [13]:
def _cache_path(rel_path):
    h = hashlib.md5(rel_path.encode()).hexdigest()
    return CACHE_DIR / f"{h}.npy"

def get_tensor_cached(rel_path):
    cp = _cache_path(rel_path)
    if cp.exists():
        return np.load(cp)
    raw, _ = wfdb.rdsamp(str(DATA_DIR / rel_path))
    clean = preprocess_ecg(raw)
    if clean is None:
        return None
    tensor = ecg_to_tensor(clean)
    np.save(cp, tensor)
    return tensor

def build_split(df_split):
    Xs, Ys = [], []
    for _, row in df_split.iterrows():
        t = get_tensor_cached(row["filename_lr"])
        if t is None:
            continue
        Xs.append(t)
        Ys.append(row["target_vec"])
    return np.stack(Xs).astype(np.float32), np.stack(Ys).astype(np.float32)

X_train, Y_train = build_split(train_df)
X_val,   Y_val   = build_split(val_df)
X_test,  Y_test  = build_split(test_df)

print("Formas:", X_train.shape, X_val.shape, X_test.shape)


Formas: (17084, 33, 64, 12) (2146, 33, 64, 12) (2158, 33, 64, 12)


## 8. Arquitectura: CNN de tres bloques


In [18]:
def build_model_deep(input_shape, n_classes):
    inputs = layers.Input(shape=input_shape)
    x = layers.Conv2D(32, 3, padding="same", activation="relu", kernel_regularizer=l2(L2_REG))(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Conv2D(64, 3, padding="same", activation="relu", kernel_regularizer=l2(L2_REG))(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Conv2D(128, 3, padding="same", activation="relu", kernel_regularizer=l2(L2_REG))(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)

    x = layers.Dense(64, activation="relu", kernel_regularizer=l2(L2_REG))(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(n_classes, activation="sigmoid")(x)

    return models.Model(inputs, outputs)

model = build_model_deep(X_train.shape[1:], len(SUPER_CLASSES))
model.summary()


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)           │ (None, 33, 64, 12)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_3 (Conv2D)                    │ (None, 33, 64, 32)          │           3,488 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_3                │ (None, 33, 64, 32)          │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 16, 32, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_4 (Dropout)                  │ (None, 16, 32, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_4 (Conv2D)                    │ (None, 16, 32, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_4                │ (None, 16, 32, 64)          │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_3 (MaxPooling2D)       │ (None, 8, 16, 64)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_5 (Dropout)                  │ (None, 8, 16, 64)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_5 (Conv2D)                    │ (None, 8, 16, 128)          │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_5                │ (None, 8, 16, 128)          │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d_1           │ (None, 128)                 │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_6 (Dropout)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 64)                  │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_7 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 5)                   │             325 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 105,317 (411.39 KB)

 Trainable params: 104,869 (409.64 KB)

 Non-trainable params: 448 (1.75 KB)

## 9. Balanceo de clases (pos_weight + focal loss)

`pos_weight` se calcula de forma inversamente proporcional a la frecuencia de cada clase en entrenamiento, acotado entre 1 y 10 para que ninguna clase minoritaria domine el gradiente. La focal loss (γ=1.0) modula además el peso de cada ejemplo según su dificultad, igual que en tu modelo principal.

In [19]:
pos_counts = Y_train.sum(axis=0)
neg_counts = len(Y_train) - pos_counts
pos_weight = np.clip(neg_counts / np.maximum(pos_counts, 1), 1.0, 10.0)
print("pos_weight por clase:", dict(zip(SUPER_CLASSES, pos_weight.round(2))))

pos_weight_tf = tf.constant(pos_weight, dtype=tf.float32)

def focal_weighted_bce(y_true, y_pred, gamma=GAMMA):
    """Entropía cruzada binaria ponderada por clase (pos_weight) y modulada por dificultad (focal, gamma)."""
    eps = 1e-7
    y_pred = tf.clip_by_value(y_pred, eps, 1 - eps)
    pos_term = pos_weight_tf * y_true * tf.pow(1 - y_pred, gamma) * tf.math.log(y_pred)
    neg_term = (1 - y_true) * tf.pow(y_pred, gamma) * tf.math.log(1 - y_pred)
    return tf.reduce_mean(-(pos_term + neg_term))

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss=focal_weighted_bce,
              metrics=[tf.keras.metrics.AUC(name="auc", multi_label=True)])


pos_weight por clase: {'NORM': 1.25, 'MI': 2.9, 'STTC': 3.08, 'HYP': 7.06, 'CD': 3.37}


## 10. Entrenamiento

Ya compilado con `focal_weighted_bce` en el apartado anterior. Umbral de decisión todavía fijo en 0.5 — eso se optimiza en 05.

In [20]:
cbs = [
    callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=8, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor="val_auc", mode="max", factor=0.5, patience=4, min_lr=1e-6),
    callbacks.ModelCheckpoint(str(MODEL_DIR / f"{NOTEBOOK_NAME}.keras"), monitor="val_auc",
                               mode="max", save_best_only=True),
]

history = model.fit(
    X_train, Y_train,
    validation_data=(X_val, Y_val),
    epochs=60,
    batch_size=32,
    callbacks=cbs,
    verbose=1,
)


Epoch 1/60
534/534 ━━━━━━━━━━━━━━━━━━━━ 42s 70ms/step - auc: 0.7918 - loss: 0.4353 - val_auc: 0.8376 - val_loss: 0.4143 - learning_rate: 0.0010
Epoch 2/60
534/534 ━━━━━━━━━━━━━━━━━━━━ 42s 71ms/step - auc: 0.8393 - loss: 0.3920 - val_auc: 0.8580 - val_loss: 0.4046 - learning_rate: 0.0010
Epoch 3/60
534/534 ━━━━━━━━━━━━━━━━━━━━ 43s 75ms/step - auc: 0.8531 - loss: 0.3775 - val_auc: 0.8628 - val_loss: 0.3975 - learning_rate: 0.0010
Epoch 4/60
534/534 ━━━━━━━━━━━━━━━━━━━━ 40s 73ms/step - auc: 0.8620 - loss: 0.3667 - val_auc: 0.8661 - val_loss: 0.4293 - learning_rate: 0.0010
Epoch 5/60
534/534 ━━━━━━━━━━━━━━━━━━━━ 42s 74ms/step - auc: 0.8677 - loss: 0.3596 - val_auc: 0.8661 - val_loss: 0.3968 - learning_rate: 0.0010
Epoch 6/60
534/534 ━━━━━━━━━━━━━━━━━━━━ 39s 73ms/step - auc: 0.8731 - loss: 0.3532 - val_auc: 0.8709 - val_loss: 0.3587 - learning_rate: 0.0010
Epoch 7/60
534/534 ━━━━━━━━━━━━━━━━━━━━ 39s 73ms/step - auc: 0.8755 - loss: 0.3501 - val_auc: 0.8623 - val_loss: 0.4091 - learning_rate:

## 11. Evaluación sobre el conjunto de test

In [21]:
def compute_metrics(Y_true, Y_pred, Y_prob):
    """Accuracy, Precision, Recall, F1 y MAE por clase, más la fila Macro (media simple)."""
    rows = []
    for i, c in enumerate(SUPER_CLASSES):
        yt, yp, ypr = Y_true[:, i], Y_pred[:, i], Y_prob[:, i]
        tp = np.sum((yt == 1) & (yp == 1))
        tn = np.sum((yt == 0) & (yp == 0))
        fp = np.sum((yt == 0) & (yp == 1))
        fn = np.sum((yt == 1) & (yp == 0))
        acc  = (tp + tn) / len(yt)
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
        mae  = np.mean(np.abs(ypr - yt))
        rows.append([CLASS_MAP_HUMAN[c], acc, prec, rec, f1, mae])

    metrics_df = pd.DataFrame(rows, columns=["Clase", "Accuracy", "Precision", "Recall", "F1", "MAE"])
    macro = metrics_df[["Accuracy", "Precision", "Recall", "F1", "MAE"]].mean()
    metrics_df.loc[len(metrics_df)] = ["Macro"] + macro.tolist()
    return metrics_df

test_probs = model.predict(X_test, verbose=0)
test_pred  = (test_probs >= 0.5).astype(int)

metrics_df = compute_metrics(Y_test, test_pred, test_probs)
metrics_df


,Clase,Accuracy,Precision,Recall,F1,MAE
0,Normal,0.846617,0.785714,0.902388,0.840019,0.245408
1,Infarto,0.806766,0.599700,0.727273,0.657354,0.319964
2,Isquemia/ST-T,0.823911,0.592398,0.867562,0.704050,0.266248
3,Hipertrofia,0.735867,0.276812,0.729008,0.401261,0.326448
4,Conducción (bloqueos),0.841520,0.633681,0.735887,0.680970,0.274088
5,Macro,0.810936,0.577661,0.792424,0.656731,0.286431


## 12. Registro en la tabla comparativa común


In [22]:
macro_row = metrics_df[metrics_df["Clase"] == "Macro"].drop(columns="Clase").iloc[0]
result_row = pd.DataFrame([{
    "Notebook": NOTEBOOK_NAME,
    "Preprocesado": PREP_VERSION,
    "Parametros_entrenables": model.count_params(),
    **macro_row.to_dict(),
}])

if COMPARISON_CSV.exists():
    comparison = pd.read_csv(COMPARISON_CSV)
    comparison = comparison[comparison["Notebook"] != NOTEBOOK_NAME]
    comparison = pd.concat([comparison, result_row], ignore_index=True)
else:
    comparison = result_row

comparison.to_csv(COMPARISON_CSV, index=False)
comparison


,Notebook,Preprocesado,Parametros_entrenables,Accuracy,Precision,Recall,F1,MAE
0,02_cnn_simple_con_filtrado,con_filtrado,3653,0.829472,0.733107,0.387682,0.457371,0.241108
1,03_cnn_profunda_con_filtrado,con_filtrado,105317,0.854310,0.733758,0.591347,0.630620,0.178831
2,04_cnn_profunda_balanceo,con_filtrado,105317,0.810936,0.577661,0.792424,0.656731,0.286431
